---
#### *Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026*
---

# Deploy Agent for Local Inference

![](../../assets/local_deployment_architecture.png)

## Deploy as code

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import mlflow
from mlflow.entities import SpanType

load_dotenv()

openai_client = OpenAI(
    api_key=os.environ["GEMINI_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

MODEL = "gemini-2.5-flash-lite"
EXPERIMENT_NAME = os.environ.get("EXPERIMENT_4_NAME", "mlflow-agent-4")

if not os.getenv("MLFLOW_TRACKING_URI"):
    mlflow.set_tracking_uri("http://127.0.0.1:5000")

experiment = mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.openai.autolog()

In [ ]:
# Create active agent model
mlflow.set_active_model(name="mlflow-agent-v1")

## Log Model

Once the active model is set, we then are going to log the code model as an MLflow model

In [ ]:
#Completions Agent
with mlflow.start_run():
    info = mlflow.pyfunc.log_model(
        name="mlflow_agent",
        python_model="mlflow_langgraph_agent.py",
        pip_requirements=["openai", "mlflow"],
    )

In [ ]:
with mlflow.start_run():
    info = mlflow.pyfunc.log_model(
        name="mlflow_langgraph_agent",
        python_model="mlflow_langgraph_agent.py",
        pip_requirements=["langchain", "langgraph", "langchain_google_genai", "mlflow"],
    )

Now that we've logged the model successfuly, we need to submit it to the 'Model Registry' in order to deploy it to a local server. We register the agent with the classic `mlflow.register_model` function that we're used to from traditional ML.

*We can also perform this action from the UI*

In [ ]:
mlflow.register_model(model_uri=info.model_uri, name="mlflow_langgraph_agent")

## A little about MLflow client

In [ ]:
# Check the model is registerd
from mlflow import MlflowClient

client = MlflowClient(tracking_uri="http://127.0.0.1:5000")

# List all registered models
for rm in client.search_registered_models():
    print(rm.name)
    for v in rm.latest_versions:
        print(f"  version {v.version}  (run_id={v.run_id}, status={v.status})")
        print(f"  models:/{rm.name}/{v.version}")

In [ ]:
mv = client.get_model_version(name="mlflow_langgraph_agent", version=1)

In [ ]:
mv

### Need to set environment variables appropriately

In [ ]:
mlflow models serve -m models:/mlflow-assistant/2 -p 5678 --env-manager=local

In [ ]:
mlflow models serve -m models:/mlflow_langgraph_agent/1 -p 5678 --env-manager=local

In [ ]:
# Perform model inference
import json
import requests

payload = json.dumps({
    "inputs": {"messages": [{"role": "user", "content": "What is MLflow?"}]},
    "params": {
        "temperature": 0.5,
        "max_tokens": 20,
    },
})
response = requests.post(
    url=f"http://127.0.0.1:5678/invocations",
    data=payload,
    headers={"Content-Type": "application/json"},
)
print(response.json())

In [ ]:
# If you need to stop the server, run in terminal:
kill -9 $(lsof -t -i:5678)